In [1]:
pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 43.2 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 19.0.1
    Uninstalling pyarrow-19.0.1:
      Successfully uninstalled pyarrow-19.0.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
pylibcudf-cu12 25.2.2 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompatible.
cudf-cu12 25.2.2 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompatible.
bigframes 2.12.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.
cudf-polars-cu12 25.6.0 requires pylibcudf-cu12=

In [2]:
# --- Imports ---
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
import gc
from tqdm import tqdm
import numpy as np
from datasets import load_dataset
import os
import evaluate
# --- Configuration ---
INPUT_FILE = "/kaggle/input/kmeaans-extracted-dataset/final_extracted_sentences.csv"
OUTPUT_FILE = "finetuned20_generated_summaries_with_metrics.csv"
ORIGINAL_DATASET_HF = "datht/FINDSum"

# Limit to 200 documents (per your previous setting, adjust to 2000 if needed)
DOC_LIMIT = 2000

# --- MODEL CONFIGURATION ---
# Attempt to automatically find your attached Kaggle model
KAGGLE_MODEL_PATH = "/kaggle/input/finetuned-led/transformers/default/2/final_financial_led" 

if not os.path.exists(KAGGLE_MODEL_PATH):
    # Fallback search
    for root, dirs, files in os.walk("/kaggle/input"):
        if "finetuned-led" in root:
            KAGGLE_MODEL_PATH = root
            break

print(f"Using Fine-Tuned Model at: {KAGGLE_MODEL_PATH}")

MODELS_TO_RUN = {
    "Fine-Tuned LED": KAGGLE_MODEL_PATH,     # Your custom model
    "LED-Base": "allenai/led-base-16384",    # Standard LED (Zero-shot)
    "BART-Baseline": "facebook/bart-large-cnn" # Standard Baseline
}

def main():
    print("="*60)
    print(f"   FINDSUM FINAL GENERATION (Docs: {DOC_LIMIT})")
    print("="*60)

    # --- 1. Load & Prep Content ---
    if not os.path.exists(INPUT_FILE):
        print(f"❌ Error: {INPUT_FILE} not found. Please run the extraction script first.")
        return

    print("[1/5] Loading extracted content...")
    df_extracted = pd.read_csv(INPUT_FILE)
    
    # Sort by rank to ensure most important sentences come first
    df_extracted.sort_values(['doc_id', 'importance_rank', 'sentence_index'], inplace=True)
    
    print("Reconstructing documents...")
    # Combine extracted sentences into one long string per document
    df_docs = df_extracted.groupby('doc_id')['sentence'].apply(lambda x: " ".join(str(s) for s in x)).reset_index()
    df_docs.columns = ['doc_id', 'source_text']
    
    # Apply Limit
    if DOC_LIMIT:
        df_docs = df_docs.head(DOC_LIMIT)
        print(f"⚠️ Processing first {DOC_LIMIT} documents.")
    
    # --- 2. Load Ground Truth (for Metrics) ---
    print(f"[2/5] Fetching Reference Summaries...")
    try:
        dataset = load_dataset(ORIGINAL_DATASET_HF, split="train", streaming=True)
        
        target_ids = set(df_docs['doc_id'].values)
        max_id = max(target_ids) if target_ids else 0
        
        id_to_summary = {}
        if max_id > 0:
            print("Streaming dataset to find matching summaries...")
            for i, example in tqdm(enumerate(dataset), total=max_id+1):
                if i in target_ids:
                    id_to_summary[i] = example['summary']
                if i >= max_id:
                    break
        
        df_docs['reference_summary'] = df_docs['doc_id'].map(id_to_summary)
        
    except Exception as e:
        print(f"Warning: Could not load references ({e}). Skipping metrics.")
        df_docs['reference_summary'] = ""

    # Initialize Metrics
    try:
        rouge = evaluate.load("rouge")
        bleu = evaluate.load("bleu")
    except Exception as e:
        print(f"Metrics load failed: {e}")
        return
    
    all_results = []

    # --- 3. Model Loop ---
    device = 0 if torch.cuda.is_available() else -1
    print(f"Inference Device: {'GPU' if device == 0 else 'CPU'}")

    for display_name, checkpoint in MODELS_TO_RUN.items():
        print(f"\n--- Processing with {display_name} ---")
        
        # Check validity of path vs Hugging Face ID
        # If it looks like a path (starts with /), check if it exists
        if checkpoint.startswith("/") and not os.path.exists(checkpoint):
            print(f"❌ Error: Local model path not found: {checkpoint}")
            print("Did you attach the model in the right sidebar?")
            continue

        try:
            # Load Pipeline
            summarizer = pipeline(
                "summarization", 
                model=checkpoint, 
                tokenizer=checkpoint, 
                device=device,
                truncation=True,
                torch_dtype=torch.float16 if device == 0 else torch.float32
            )
        except Exception as e:
            print(f"❌ Failed to load {display_name}: {e}")
            continue

        model_generations = []
        
        # Generation Loop
        for i, row in tqdm(df_docs.iterrows(), total=len(df_docs), desc=f"Gen {display_name}"):
            try:
                # Parameters customization based on model architecture
                # Both LED versions (Fine-Tuned and Base) use the same parameters
                if "LED" in display_name or "led" in str(checkpoint).lower():
                    gen_kwargs = {
                        "max_length": 512, 
                        "min_length": 100, 
                        "no_repeat_ngram_size": 3, 
                        "num_beams": 4,
                        "length_penalty": 2.0
                    }
                else:
                    # BART (Standard 1024 token model)
                    gen_kwargs = {
                        "max_length": 400, 
                        "min_length": 80, 
                        "no_repeat_ngram_size": 3, 
                        "num_beams": 4,
                        "length_penalty": 2.0
                    }

                output = summarizer(row['source_text'], do_sample=False, **gen_kwargs)
                summary = output[0]['summary_text']
                
                model_generations.append(summary)
                
                all_results.append({
                    'doc_id': row['doc_id'],
                    'model': display_name,
                    'generated_summary': summary,
                    'reference_summary': row['reference_summary']
                })
                
            except Exception as e:
                print(f"Error doc {row['doc_id']}: {e}")
                model_generations.append("") 

            # Incremental Save
            if len(all_results) % 50 == 0:
                pd.DataFrame(all_results).to_csv(OUTPUT_FILE, index=False)

        # --- 4. Compute Metrics ---
        print(f"Computing metrics for {display_name}...")
        valid_preds = [p for p in model_generations if p]
        valid_refs = [r for p, r in zip(model_generations, df_docs['reference_summary']) if p and r]
        
        if valid_preds and valid_refs:
            r_scores = rouge.compute(predictions=valid_preds, references=valid_refs)
            b_scores = bleu.compute(predictions=valid_preds, references=valid_refs)
            
            print(f"📊 {display_name} Scores:")
            print(f"   ROUGE-1: {r_scores['rouge1']:.4f}")
            print(f"   ROUGE-2: {r_scores['rouge2']:.4f}")
            print(f"   ROUGE-L: {r_scores['rougeL']:.4f}")
            print(f"   BLEU:    {b_scores['bleu']:.4f}")
        
        # Cleanup
        del summarizer
        gc.collect()
        torch.cuda.empty_cache()

    # --- 5. Final Save ---
    print("\n[5/5] Saving Final Results...")
    if all_results:
        df_final = pd.DataFrame(all_results)
        df_final.to_csv(OUTPUT_FILE, index=False)
        print(f"✅ Saved to: {OUTPUT_FILE}")
    else:
        print("⚠️ No results generated.")

if __name__ == "__main__":
    main()

2025-12-01 16:30:46.969982: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764606647.118407      20 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764606647.158787      20 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...

Error doc 1775: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.

Error doc 1776: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.

Error doc 1777: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.

Error doc 1778: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously 

Gen BART-Baseline: 100%|██████████| 2000/2000 [01:00<00:00, 32.98it/s] 

Error doc 1929: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.

Error doc 1930: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.

Error doc 1931: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.

Error doc 1932: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously 

📊 BART-Baseline Scores:
   ROUGE-1: 0.2010
   ROUGE-2: 0.0511
   ROUGE-L: 0.1211
   BLEU:    0.0066


RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
